# DEM (Elevation) Analysis

**Purpose:** download SRTM elevation tiles covering the GPS data's bounding box, merge them into a single GeoTIFF, and visualize it. The merged DEM is used elsewhere to add grade/slope as a covariate for haul-segment travel time.

In [ ]:
import sys
sys.path.append("../..")

import glob
import matplotlib.pyplot as plt
import rasterio

from gps_lib import io_utils, dem, config

In [ ]:
gps_df = io_utils.load_gps_data("gps_data.csv")
min_lat, max_lat = gps_df["lat"].min(), gps_df["lat"].max()
min_lng, max_lng = gps_df["lng"].min(), gps_df["lng"].max()
min_lat, max_lat, min_lng, max_lng

In [ ]:
tile_dir = str(config.DATA_DIR / "dem_tiles")
dem.download_srtm_coverage(min_lat, max_lat, min_lng, max_lng, tile_dir)

In [ ]:
dem.unzip_tiles(tile_dir)

sample_tile = glob.glob(f"{tile_dir}/*.hgt")[0]
with rasterio.open(sample_tile) as src:
    plt.imshow(src.read(1), cmap="terrain")
    plt.colorbar()
    plt.title("SRTM DEM (sample tile)")
    plt.show()

In [ ]:
out_path = str(config.DATA_DIR / "dem_merged.tif")
dem.merge_dem_tiles(tile_dir, out_path)
print("Saved:", out_path)

In [ ]:
with rasterio.open(out_path) as src:
    elevation = src.read(1)
    bounds = src.bounds

plt.figure(figsize=(8, 6))
plt.imshow(elevation, cmap="terrain", extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
plt.colorbar(label="Elevation (m)")
plt.title("Merged DEM")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()